# PDFVQA Dataset Downloader

Downloads the [PDFVQA Kaggle competition](https://www.kaggle.com/competitions/pdfvqa/data) dataset.

**Prerequisites:** You need a Kaggle account and API token.
1. Go to https://www.kaggle.com/settings → API → *Create New Token*
2. This downloads `kaggle.json` containing `{"username": "...", "key": "..."}`
3. Accept the competition rules at https://www.kaggle.com/competitions/pdfvqa/rules (required before download)
4. Either paste credentials in the cell below, or upload `kaggle.json`

In [ ]:
# Install kaggle CLI
!pip install -q kaggle

In [ ]:
import os, json, pathlib

# ── Option A: paste credentials directly ────────────────────────────────────
KAGGLE_USERNAME = ""   # e.g. "johndoe"
KAGGLE_KEY      = ""   # e.g. "abc123..."

# ── Option B: point to an uploaded kaggle.json ──────────────────────────────
KAGGLE_JSON_PATH = ""  # e.g. "/teamspace/studios/this_studio/kaggle.json"

# ── Setup ────────────────────────────────────────────────────────────────────
kaggle_dir = pathlib.Path.home() / ".kaggle"
kaggle_dir.mkdir(exist_ok=True)
kaggle_json = kaggle_dir / "kaggle.json"

if KAGGLE_USERNAME and KAGGLE_KEY:
    creds = {"username": KAGGLE_USERNAME, "key": KAGGLE_KEY}
    kaggle_json.write_text(json.dumps(creds))
    kaggle_json.chmod(0o600)
    print("Credentials written from username/key.")
elif KAGGLE_JSON_PATH and os.path.exists(KAGGLE_JSON_PATH):
    import shutil
    shutil.copy(KAGGLE_JSON_PATH, kaggle_json)
    kaggle_json.chmod(0o600)
    print(f"Credentials copied from {KAGGLE_JSON_PATH}")
elif kaggle_json.exists():
    print("Found existing ~/.kaggle/kaggle.json — using it.")
else:
    raise RuntimeError(
        "No Kaggle credentials found. Fill in KAGGLE_USERNAME/KAGGLE_KEY "
        "or set KAGGLE_JSON_PATH above."
    )

In [ ]:
# Verify authentication
!kaggle competitions list --search pdfvqa 2>&1 | head -5

In [2]:
import pathlib

# Where to put the downloaded files (same dir as the baseline notebook)
DOWNLOAD_DIR = pathlib.Path("/teamspace/studios/this_studio/src/part2/pdfvqa_prep_work")


In [ ]:
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

print(f"Download destination: {DOWNLOAD_DIR}")

In [ ]:
# Download competition zip, then extract
# NOTE: you must have accepted the competition rules on kaggle.com first
import zipfile, pathlib

!kaggle competitions download -c pdfvqa -p "{DOWNLOAD_DIR}"

# Find and extract any zip files produced
zips = list(DOWNLOAD_DIR.glob("*.zip"))
if not zips:
    print("No zip file found — files may already be extracted.")
else:
    for z in zips:
        print(f"Extracting {z.name} ...")
        with zipfile.ZipFile(z) as zf:
            zf.extractall(DOWNLOAD_DIR)
        z.unlink()   # remove zip after extraction
        print(f"  done, removed {z.name}")
print("All done.")

In [ ]:
# List what was downloaded
import os
for root, dirs, files in os.walk(DOWNLOAD_DIR):
    # skip hidden dirs
    dirs[:] = [d for d in dirs if not d.startswith('.')]
    level = root.replace(str(DOWNLOAD_DIR), '').count(os.sep)
    indent = '  ' * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = '  ' * (level + 1)
    for f in files:
        size = os.path.getsize(os.path.join(root, f))
        print(f"{subindent}{f}  ({size/1024/1024:.1f} MB)")

In [1]:
# Quick sanity-check: load a CSV and a pickle
import pandas as pd, pickle

train_df = pd.read_csv(DOWNLOAD_DIR / "train_dataframe.csv")
print("train_dataframe.csv:", train_df.shape)
print(train_df.head(3))

with open(DOWNLOAD_DIR / "train_doc_info.pkl", "rb") as f:
    train_doc_info = pickle.load(f)
print(f"\ntrain_doc_info: {len(train_doc_info)} documents")

NameError: name 'DOWNLOAD_DIR' is not defined